# Attention ablations: does every Transformer component earn its place on small data?

Runs the full study end to end on a Colab GPU.

**Before you start:** Runtime, Change runtime type, T4 GPU.

The whole study is roughly 10 configurations x 3 seeds. On a T4 with 20,000 training
examples expect somewhere around 45 to 90 minutes. Start with the `small` run if you
want to see the shape of the results first.

## 1. Install and clone

In [ ]:
!git clone https://github.com/YOUR_USERNAME/attention-ablations.git
%cd attention-ablations
!pip install -q -r requirements.txt

## 2. Confirm the implementation is correct

These are the numerical equivalence tests against PyTorch's own attention. If any of these fail, nothing below is worth reading.

In [ ]:
!PYTHONPATH=src python -m pytest tests/ -v --tb=short

## 3. Check the GPU is actually being used

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Offline smoke test

No download, no GPU needed. Confirms the pipeline runs before committing to the long job.

In [ ]:
!python scripts/run_ablations.py --synthetic --epochs 4 --seeds 2 \
    --patience 0 --d-model 32 --d-ff 64 --out results/smoke

## 5. The small run

1,000 training examples, matching the size of the original coursework notebook. This is the
comparison that shows what changed. About 5 to 10 minutes on a T4.

In [ ]:
!python scripts/run_ablations.py --n-train 1000 --n-val 500 --n-test 500 \
    --epochs 20 --seeds 3 --out results/small

## 6. The data-scaling sweep, which is the headline result

Trains the model at 500, 1,000, 2,000, 5,000, 10,000 and 20,000 examples and fits tf-idf
baselines on the same subsets. Reports the crossover point: the training set size at which
the Transformer overtakes the linear model.

Validation and test sets stay identical at every size, and the vocabulary is rebuilt from
each training subset so the small-data runs do not borrow vocabulary they would not have had.

Roughly 30 to 60 minutes on a T4. Run `scaling-quick` first if you want a look sooner.

In [ ]:
# A first look: 3 sizes, 1 seed, about 10 minutes.
!python scripts/run_scaling.py --sizes 500 2000 10000 \
    --epochs 20 --seeds 1 --batch-size 64 --out results/scaling-quick

In [ ]:
# The real sweep: 6 sizes, 3 seeds.
!python scripts/run_scaling.py --sizes 500 1000 2000 5000 10000 20000 \
    --epochs 30 --seeds 3 --batch-size 64 --out results/scaling

### The scaling curve and the crossover point

In [ ]:
from IPython.display import Image, display

display(Image("results/scaling/scaling_curve.png"))
display(Image("results/scaling/scaling_gap.png"))
print(open("results/scaling/scaling_table.md").read())

## 7. The full ablation run

20,000 training examples, the full 25,000 example test set, 3 seeds per configuration.
This is the run whose table goes in the README.

In [ ]:
!python scripts/run_ablations.py --n-train 20000 --n-val 5000 --n-test 25000 \
    --epochs 30 --seeds 3 --batch-size 64 --out results/full

## 8. Look at the ablation figures

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run = Path("results/full")
for name in ["ablations.png", "training_curves.png", "attention_heads.png"]:
    path = run / name
    if path.exists():
        print(name)
        display(Image(str(path)))

## 9. The ablation table, ready to paste into the README

In [ ]:
print(open("results/full/results_table.md").read())

## 10. Which tokens the model actually attends to

The claim in the original coursework notebook was that heads focus on sentiment-bearing
words. This is where that gets checked rather than asserted.

In [ ]:
print(open("results/full/attention_ranking.txt").read())

## 11. Commit the results back

Figures, `results.json` and the tables are worth committing. Model weights are not, and
`.gitignore` already excludes `results/*.pt`.

Set your name and email first, and use a personal access token when prompted for a password.

In [ ]:
!git config user.name "YOUR_NAME"
!git config user.email "YOUR_EMAIL"
!git add results/ && git commit -m "Add results from full IMDB run" && git push